In [ ]:
av_raw = session.table(f"{DB}.{RAW}.STREAM_T_ALLOWABLE_VALUES").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
# All transformations in single .select()
ind_cols = ["VALUE_DEFAULT_IND"]
code_cols = ["VALUE_CODE"]
desc_cols = ["VALUE_SHORT_DESC", "VALUE_LONG_DESC"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["MINIMUM_NUM", "MAXIMUM_NUM", "SOURCE_FILE_NAME"]

all_cols = [c.name for c in av_raw.schema.fields]
select_exprs = []
for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in ind_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c))
    elif c in code_cols:
        select_exprs.append(upper(trim(col(c))).alias(c))
    elif c in desc_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("NA")).otherwise(trim(col(c))), lit("NA")).alias(c))
    elif c in user_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c))
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

av_clean = av_raw.select(select_exprs)
print("Transformations applied")

In [ ]:
valid_av = av_clean.filter(col("AV_ID").is_not_null())
invalid_av = av_clean.filter(col("AV_ID").is_null()).with_column("ERROR_REASON", lit("AV_ID_NULL"))

av_final = valid_av.with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())
av_final.write.save_as_table(f"{DB}.{STG}.{STG_ALLOWABLE_VALUES}", mode="truncate")
print(f"AV loaded into {STG}.{STG_ALLOWABLE_VALUES}")

In [ ]:
invalid_count = invalid_av.count()

if invalid_count > 0:
    invalid_av.select(
        lit("T_ALLOWABLE_VALUES").alias("TABLE_NAME"),
        col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"),
        col("RAW_LOAD_TIMESTAMP"),
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_ALLOWABLE_VALUES}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_ALLOWABLE_VALUES_LOAD', 'STAGING',
    datetime.now(), datetime.now(),
    rows_processed, invalid_count, status, None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_ALLOWABLE_VALUES_LOAD', 'STAGING',
    f'ALLOWABLE_VALUES staging completed. Rows: {rows_processed}, Failed: {invalid_count}'
)
print(f"Done | Valid: {rows_processed} | Invalid: {invalid_count}")